# TD Neuro-symbolique: Mini-NeurASP
Cours d'IA Hybride et Apprentissage Automatique (L6SOIAAU), Licence « Sciences du langage » (L3)  
UFR Sociologie et Informatique pour les Sciences Humaines, Sorbonne Université   
François OLIVIER

In [ ]:
# Packages nécessaires pour ce TD. Décommentez pour installer (une seule fois): 
#!pip install tensorflow matplotlib clingo

## Introduction 

On considère une **grille 3×3 manuscrite**, dont le remplissage doit respecter les **contraintes** suivantes (version simplifiée du sudoku) :

$C_1~$ :  Chaque cellule contient un chiffre parmi {1, 2, 3}  
$C_2~$ :  Chaque ligne contient exactement une fois 1, 2 et 3  
$C_3~$ :  Chaque colonne contient exactement une fois 1, 2 et 3  

**Modélisation**: Ces contraintes en **langage naturel** devront être **traduites en règles symboliques (ASP)**.  
En revanche, les chiffres présents dans la grille sont des **images MNIST**, et devront  être **traités par un modèle neuronal** avant de pouvoir alimenter le raisonnement symbolique.

L’objectif est donc de **combiner perception neuronale et raisonnement symbolique pour résoudre la grille**.

Les grilles sur lesquelles nous allons travailler sont données ci-dessous : une **grille facile** et une **grille difficile**. Il s’agit de **grilles d’indices** (listes de listes Python), où chaque valeur correspond à l’indice d’une image dans le **jeu de test** de MNIST.

Faites tourner les deux cellules suivantes pour visualiser ces grilles.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

(_,_),(x_test,_) = tf.keras.datasets.mnist.load_data()

def show_sudoku(grid):
    fig, axes = plt.subplots(3, 3, figsize=(3,3))
    for i in range(3):
        for j in range(3):
            ax = axes[i, j]
            idx = grid[i][j]
            if idx is not None:
                ax.imshow(x_test[idx], cmap='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(True)
    plt.show()

In [ ]:
index_grid_easy = [[32,    None,  None], 
                   [None,  37,    None], 
                   [None,  None,  174]]  

index_grid_hard = [[2927,  2,     1], 
                   [None,  None,  None], 
                   [180,   None,  None]]  
show_sudoku(index_grid_easy)
show_sudoku(index_grid_hard)

> **Exercice** :
> 1) Trouvez la solution de ces deux grilles.
> 2) Dans la deuxième grille, le chiffre en haut à gauche est difficile à lire. Pourtant, il est possible de s'assurer de sa valeur en s’appuyant sur les **autres chiffres déjà présents** et sur les **contraintes logiques** de la grille. Faites dès à présent le parallèle avec le rôle que joueront les différents composants du système que vous allez construire.

## 1) Composant Neuronal

Dans cette section, nous allons construire un composant neuronal capable de prédire les chiffres à partir des images.

### 1.1) Charger et inspecter le dataset complet

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

> **Exercice** :
>
> 1) Afficher le type de `x_train` et importer le package nécessaire pour manipuler ce type d’objets.
> 2) Les images ont-elles plusieurs channels (comme RGB) ou un seul ? Vérifiez avec la shape de `x_train`
> 2) Déterminer combien d'exemples contiennent `x_train`, `y_train`, `x_test`, `y_test`  
> 3) Afficher l'image du premier chiffre du dataset avec `matplotlib`  
>
>    *Rappel :* une image MNIST se récupère avec `x_test[i]` (ou `x_train[i]`) et s’affiche avec `plt.imshow(..., cmap='gray')`.

### 1.2) Préparer les données

In [ ]:
# Les images MNIST (valeurs entre 0 et 255) sont transformées en float normalisés entre 0 et 1.
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Ajouter une dimension pour le CNN car les couches Conv2D attendent une dimension "channel" 
x_train = x_train[..., np.newaxis]  
x_test = x_test[..., np.newaxis]

> **Exercice** :  Afficher les nouvelles shapes de `x_train`, `y_train`.   
>Est-ce que l’ajout de la dimension a changé les données ? Justifiez.
>

### 1.3) Créer, entraîner et évaluer le modèle neuronal

> **Exercice** : En gardant un maximum de **3 epochs** et **5000 exemples d’entraînement**, compléter et améliorer le modèle, notamment en répondant à ces questions et en suivant ces instructions:
>
> 1) Dans la couche `tf.keras.layers.Conv2D(16, 3, activation="relu")`, à quoi correspondent `16` et `3` ?
>
> 2) Ajouter une seconde couche `Conv2D` suivie d’une couche `MaxPool2D` au bon endroit dans le modèle.  
>    Vous pouvez faire varier le **nombre de filtres** de cette couche.
>
> *Pour rappel* : une architecture classique pour MNIST suit le schéma :
>
> `Conv2D → MaxPool → Conv2D → MaxPool → Flatten → Dense → Output`
>
> 3) Faire varier le **nombre de neurones** de la couche `Dense`.
>
> 4) (Optionnel) Tester différents **optimizers** :
>    - `"adam"`    -> Adaptive Moment Estimation (combine moyenne et variance des gradients)
>    - `"sgd"`     -> Stochastic Gradient Descent (descente de gradient simple)
>    - `"rmsprop"` -> Root Mean Square Propagation (adapte le pas selon la variance des gradients)
>
> Comparez les résultats obtenus en termes de `test accuracy`.


In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(16, 3, activation="relu"), 
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(optimizer="sgd",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(
    x_train[:5000],
    y_train[:5000],
    epochs=3,
    batch_size=64,
    validation_split=0.1)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
model.summary()


> **Exercice** :
>
> 1) Écrire une fonction `predict_123(img)` qui retourne, pour une image donnée, les probabilités associées aux chiffres {1, 2, 3}, triées par ordre décroissant. Par exemple, `[(1, '0.9922'), (3, '0.0008'), (2, '0.0001')]`.
>
> 2) Utiliser cette fonction pour prédire un des chiffres de la grille d’indices de l’énoncé.
>
> *Rappels* :
> - Vu qu'on ne fournit qu'une seule image, et que le modèle attend un **batch**, il faut toujours ajouter une dimension pour transformer l’image en batch de taille 1.
> - Les images de la grille proviennent du **jeu de test** (`x_test`).

## 2) Composant Symbolique

Dans cette section, nous allons construire un composant symbolique permettant de représenter et de contraindre les solutions possibles.

> **Exercice** : Rendez-vous sur le site https://potassco.org/clingo/run/ pour écrire un programme *clingo*.  
> Les programmes se lancent en cliquant sur `Run!` ou `Ctrl + Enter`. Choisissez le mode `enumerate all` afin d’afficher tous les modèles possibles.
>
> Construire progressivement le programme *clingo* qui décrit des grilles 3×3.
>
> Le programme doit :
>
> 1) déclarer les 9 cellules de la grille à l’aide d’un prédicat `cell(i,j)`, où `i` et `j` sont les indices de ligne et de colonne ;
>
> 2) générer toutes les grilles complètes possibles remplies avec les valeurs 1, 2 et 3, à l’aide d’un prédicat ternaire `val(i,j,d)` où `d` est la valeur de la cellule `(i,j)`    
>
>     *Indication* : chaque cellule doit recevoir exactement une valeur, utilisez pour cela une *choice rule* de la forme  
> `{option_1 ; ... ; option_n} = 1` ;
>
> 3) contraindre les grilles possibles en ajoutant des **contraintes d’intégrité** (*integrity constraints*) correspondant aux contraintes $C_2$ et $C_3$.
>
> 4) utiliser `#show` pour afficher uniquement les atomes `val/3` dans les solutions (*stable models*).

In [ ]:
from clingo import Control

> **Exercice** : 
> 1) Remplacez le programme dans le code ci-dessous par celui que vous avez écrit dans l'interface sur le site.
>
> 2) Simplifiez davantage le programme avec du *pooling* si possible. 
>
> 3) Exécutez la cellule et commentez le code pour votre compréhension.

In [ ]:
program = """
pleut.
mouille :- pleut.
"""

ctl = Control(["0"])  # le ["0"] demande à clingo de trouver toutes les solutions. 
ctl.add("base", [], program)
ctl.ground([("base", [])])

answers = []

ctl.solve(on_model=lambda m: answers.append(m.symbols(shown=True)))

if answers:
    for i, answer in enumerate(answers, 1):
        print(f"Model {i}:", answer)
else:
    print("No solution found")


## 3) Combiner les Composants

Dans cette section, nous allons combiner les deux approches afin de résoudre le problème de manière neuro-symbolique.

> **Exercice** : 
>
> 1) Écrire une fonction `fill_grid(index_grid)` qui combine les composants **neuronaux** et **symboliques** construits plus haut. Cette fonction prend en **entrée une grille d’indices**, et doit retourner en sortie la **solution complète correspondante**.
>
> *Indications pour construire la fonction* :
> - reprendre le programme *clingo* écrit dans la section précédente ;
> - parcourir la grille pour aller chercher les indices des images et utiliser la fonction `predict_123(...)` définie plus haut afin de récupérer les candidats possibles ;
> - ajouter au programme *clingo* comme un fait, pour chaque chiffre manuscrit, la prédiction la plus probable du modèle encodée ;
> - lancer *clingo* et récupérer la solution trouvée.
>
> 2) Testez votre fonction sur `index_grid_easy` et puis sur `index_grid_hard`. Que remarquez-vous pour le deuxième cas? Expliquez et ajoutez des `print` à votre code pour supporter votre justification.

> **Exercice** : 
>
> 1) Améliorer la fonction précédente afin que **la partie symbolique tienne compte des incertitudes du modèle neuronal**.
>
>   *Indications* :
>   - une possibilité consiste à construire un dictionnaire de dictionnaires stockant, pour chaque image, les chiffres prédits ainsi que leurs probabilités ;
>   - il faut ensuite associer ces probabilités aux solutions dans lesquelles ces chiffres apparaissent. Pour cela, vous pouvez utiliser la fonction `on_model` ci-dessous, à intégrer **dans** votre fonction améliorée et à appeler avec `ctl.solve(on_model=on_model)`.
>
> 2) A quel type d'architecture ce système neuro-symbolique appartient-il? Pourquoi? En une ou deux phrases, expliquez comment la force d’un composant compense les faiblesses de l’autre. 

In [ ]:
def on_model(m):
        atoms = m.symbols(shown=True)
        prob = 1.0

        for atom in atoms:
            # atom = val(row,col,d)
            if atom.name == "val":
                row = atom.arguments[0].number
                col = atom.arguments[1].number
                d   = atom.arguments[2].number

                if (row, col) in cell_probs:
                    prob *= cell_probs[(row, col)][d]

        answers.append((atoms, prob))